

- BUild the BAsic chatboot for the based on the user query return the answer give structed the answer.

- use  this chatbot used the advanced method of langgraph and build the chatbot used the chatting ,RAG ,Action ,UI ,menory and langsmith 

- used the this chatbot memory concept for build the memnory ,persistence human in the loop and retry concept

- 

In [3]:
import os 
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver




In [4]:
load_dotenv()

True

In [5]:

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
llm=ChatGoogleGenerativeAI( model="gemini-2.5-flash",api_key=os.getenv("GOOGLE_API_KEY"))

def chat_node(state:ChatState):

    # take user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages)

    # response store state
    return {'messages': [response]}





In [7]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [8]:
# this is ask the only one question not memeoty rememeber 
#inital_state={'messages':[HumanMessage(content='what is the capital of india')]}

#chatbot.invoke(inital_state)['messages'][-1].content

In [9]:
thread_id='1'

current_state = {'messages': []}

print("Chatbot Initialized! Type 'exit' to quit.\n")
while True:
    user_message = input('Type here: ')
    print('User:', user_message)
    
    # Check for exit commands
    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break
        
    # FIX: Skip the iteration if the user typed nothing or only spaces
    if not user_message.strip():
        print("AI: Please type a valid message.")
        continue

    config={'configurable':{'thread_id':thread_id}}

    # Invoke the chatbot safely
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]},config=config)
    print('AI:', response['messages'][-1].content)


Chatbot Initialized! Type 'exit' to quit.

User: hi my name is shubham i am 24 year old
AI: Hi Shubham, nice to meet you! Thanks for introducing yourself. How can I help you today?
User: wahtis my age
AI: Based on what you just told me, you are **24 years old**.
User: exit


In [10]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is shubham i am 24 year old', additional_kwargs={}, response_metadata={}, id='51d77141-2e9f-44f7-b9aa-7932e597e584'), AIMessage(content='Hi Shubham, nice to meet you! Thanks for introducing yourself. How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fd725-4ac2-7350-a6d1-eb3f733aa408-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 21, 'total_tokens': 36, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='wahtis my age', additional_kwargs={}, response_metadata={}, id='545d5921-4023-4405-a51c-348b92fba3fb'), AIMessage(content='Based on what you just told me, you are **24 years old**.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_p